In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1yqNsHCzyeIBDokFU7FazTnxi_EHKKtGX", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


# Megatron-Style Tensor-Parallel MLP Block

*Part 2 of the Vizuara Tensor Parallelism series*  
*Estimated time: 40-50 minutes*

In this notebook, you will build a **Megatron-LM style tensor-parallel MLP block** from the column-parallel and row-parallel layers we developed in Notebook 1.

You will:
1. Implement the full column-parallel -> GeLU -> row-parallel -> allreduce pipeline
2. Add residual connections and layer normalization
3. Verify mathematical correctness against a standard MLP
4. Measure memory savings at different parallelism degrees
5. Profile computation and communication patterns

**Prerequisites:** Notebook 01 (column-parallel and row-parallel linear layers).

In [ ]:
#@title 🎧 Listen: Ai Assistant Note
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_ai_assistant_note.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** -- it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/gpu-programming/tensor-parallelism/practice)**

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

torch.manual_seed(42)
print("\nSetup complete!")

---


In [ ]:
#@title 🎧 Transition: Section1 Review
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_section1_review.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 1: Building Blocks Review

Let us quickly re-implement our parallel linear layers from Notebook 1.

In [ ]:
#@title 🎧 Code Walkthrough: Parallel Layers
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_parallel_layers.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class ColumnParallelLinear(nn.Module):
    """Linear layer with weight split along columns (output dimension)."""
    
    def __init__(self, in_features: int, out_features: int, world_size: int, rank: int,
                 bias: bool = False):
        super().__init__()
        assert out_features % world_size == 0
        self.in_features = in_features
        self.out_features_per_gpu = out_features // world_size
        self.world_size = world_size
        self.rank = rank
        
        self.weight = nn.Parameter(torch.empty(in_features, self.out_features_per_gpu))
        nn.init.kaiming_uniform_(self.weight)
        
        self.bias = None
        if bias:
            self.bias = nn.Parameter(torch.zeros(self.out_features_per_gpu))
    
    def forward(self, x):
        out = x @ self.weight
        if self.bias is not None:
            out = out + self.bias
        return out
    
    @staticmethod
    def from_linear(linear: nn.Linear, world_size: int, rank: int):
        """Create from a standard nn.Linear by taking the appropriate column shard."""
        in_f = linear.in_features
        out_f = linear.out_features
        cols = out_f // world_size
        layer = ColumnParallelLinear(in_f, out_f, world_size, rank, bias=linear.bias is not None)
        # nn.Linear stores weight transposed: (out_features, in_features)
        layer.weight = nn.Parameter(linear.weight.data[rank*cols:(rank+1)*cols, :].T.clone())
        if linear.bias is not None:
            layer.bias = nn.Parameter(linear.bias.data[rank*cols:(rank+1)*cols].clone())
        return layer


class RowParallelLinear(nn.Module):
    """Linear layer with weight split along rows (input dimension)."""
    
    def __init__(self, in_features: int, out_features: int, world_size: int, rank: int,
                 bias: bool = False):
        super().__init__()
        assert in_features % world_size == 0
        self.in_features_per_gpu = in_features // world_size
        self.out_features = out_features
        self.world_size = world_size
        self.rank = rank
        
        self.weight = nn.Parameter(torch.empty(self.in_features_per_gpu, out_features))
        nn.init.kaiming_uniform_(self.weight)
        
        # Bias is only added after allreduce, so only rank 0 needs it
        # (or we can add it after allreduce)
        self.bias = None
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
    
    def forward(self, x):
        """Returns partial sum -- caller must allreduce!"""
        return x @ self.weight
    
    @staticmethod
    def from_linear(linear: nn.Linear, world_size: int, rank: int):
        """Create from a standard nn.Linear by taking the appropriate row shard."""
        in_f = linear.in_features
        out_f = linear.out_features
        rows = in_f // world_size
        layer = RowParallelLinear(in_f, out_f, world_size, rank, bias=linear.bias is not None)
        # nn.Linear stores weight transposed: (out_features, in_features)
        layer.weight = nn.Parameter(linear.weight.data[:, rank*rows:(rank+1)*rows].T.clone())
        if linear.bias is not None and rank == 0:
            layer.bias = nn.Parameter(linear.bias.data.clone())
        elif linear.bias is not None:
            layer.bias = nn.Parameter(torch.zeros_like(linear.bias.data))
        return layer


def simulated_allreduce(tensors):
    """Simulate allreduce (sum) across GPUs."""
    return sum(tensors)

print("Building blocks defined.")

---


In [ ]:
#@title 🎧 Transition: Section2 Standard Mlp
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_section2_standard_mlp.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 2: The Standard (Non-Parallel) MLP Block

A transformer MLP block computes:

$$\text{MLP}(X) = \text{GeLU}(X W_1 + b_1) W_2 + b_2$$

where $W_1 \in \mathbb{R}^{h \times 4h}$ and $W_2 \in \mathbb{R}^{4h \times h}$.

Let us build the reference implementation first.

In [ ]:
#@title 🎧 Code Walkthrough: Standard Mlp Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_standard_mlp_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class StandardMLP(nn.Module):
    """Standard (non-parallel) transformer MLP block."""
    
    def __init__(self, hidden_dim: int, intermediate_dim: int = None):
        super().__init__()
        if intermediate_dim is None:
            intermediate_dim = 4 * hidden_dim
        
        self.fc1 = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.fc2 = nn.Linear(intermediate_dim, hidden_dim, bias=False)
    
    def forward(self, x):
        z = F.gelu(self.fc1(x))  # (batch, seq, intermediate_dim)
        y = self.fc2(z)          # (batch, seq, hidden_dim)
        return y


# Create a reference MLP
h = 64
intermediate = 4 * h  # 256
batch, seq = 2, 8

torch.manual_seed(42)
mlp_ref = StandardMLP(h, intermediate).to(device)
X = torch.randn(batch, seq, h, device=device)

Y_ref = mlp_ref(X)
print(f"Input shape:  {X.shape}")
print(f"Output shape: {Y_ref.shape}")
print(f"W1 shape: {mlp_ref.fc1.weight.shape} (nn.Linear stores transposed)")
print(f"W2 shape: {mlp_ref.fc2.weight.shape}")
print(f"Total parameters: {sum(p.numel() for p in mlp_ref.parameters()):,}")

---


In [ ]:
#@title 🎧 Transition: Section3 Tensor Parallel Mlp
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_section3_tensor_parallel_mlp.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 3: Tensor-Parallel MLP Block

Now the main event. The Megatron-LM MLP applies tensor parallelism as:

1. **$W_1$: column-parallel** -- split along output dimension
2. **GeLU: local** -- element-wise, no communication
3. **$W_2$: row-parallel** -- split along input dimension
4. **Allreduce** -- sum partial results

The key insight: column-parallel output is split, which is exactly what row-parallel input needs!

In [ ]:
#@title 🎧 Code Walkthrough: Tensor Parallel Mlp Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_tensor_parallel_mlp_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TensorParallelMLP(nn.Module):
    """Megatron-LM style tensor-parallel MLP block.
    
    This represents what runs on a SINGLE GPU (one shard).
    The allreduce is simulated externally.
    """
    
    def __init__(self, hidden_dim: int, intermediate_dim: int,
                 world_size: int, rank: int):
        super().__init__()
        self.world_size = world_size
        self.rank = rank
        
        # First linear: column-parallel (split output dim)
        self.fc1 = ColumnParallelLinear(
            hidden_dim, intermediate_dim, world_size, rank
        )
        # Second linear: row-parallel (split input dim)
        self.fc2 = RowParallelLinear(
            intermediate_dim, hidden_dim, world_size, rank
        )
    
    def forward(self, x):
        """Forward pass for this GPU's shard.
        
        Args:
            x: Input of shape (batch, seq, hidden_dim) -- replicated on all GPUs
        Returns:
            Partial output of shape (batch, seq, hidden_dim) -- needs allreduce!
        """
        # Column-parallel: x @ W1_shard -> split intermediate activations
        z = self.fc1(x)        # (batch, seq, intermediate_dim // world_size)
        
        # GeLU: element-wise, no communication needed
        z = F.gelu(z)
        
        # Row-parallel: split_activations @ W2_shard -> partial sum
        y = self.fc2(z)        # (batch, seq, hidden_dim) -- partial!
        
        return y  # Caller must allreduce across all GPUs
    
    @staticmethod
    def from_standard_mlp(mlp: StandardMLP, world_size: int, rank: int):
        """Create from a standard MLP by sharding its weights."""
        h = mlp.fc1.in_features
        inter = mlp.fc1.out_features
        tp_mlp = TensorParallelMLP(h, inter, world_size, rank)
        tp_mlp.fc1 = ColumnParallelLinear.from_linear(mlp.fc1, world_size, rank)
        tp_mlp.fc2 = RowParallelLinear.from_linear(mlp.fc2, world_size, rank)
        return tp_mlp

print("TensorParallelMLP defined.")

In [ ]:
#@title 🎧 Code Walkthrough: Create Tp Mlp
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_create_tp_mlp.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Create tensor-parallel MLP from the reference
T = 2  # tensor parallelism degree

tp_mlps = [
    TensorParallelMLP.from_standard_mlp(mlp_ref, world_size=T, rank=i).to(device)
    for i in range(T)
]

print(f"Tensor parallelism degree: {T}")
print(f"\nParameters per GPU:")
for i, mlp in enumerate(tp_mlps):
    params = sum(p.numel() for p in mlp.parameters())
    print(f"  GPU {i}: {params:,} parameters")
    print(f"    fc1 weight: {mlp.fc1.weight.shape}")
    print(f"    fc2 weight: {mlp.fc2.weight.shape}")

ref_params = sum(p.numel() for p in mlp_ref.parameters())
gpu_params = sum(p.numel() for p in tp_mlps[0].parameters())
print(f"\nFull model parameters: {ref_params:,}")
print(f"Parameters per GPU:    {gpu_params:,} ({100*gpu_params/ref_params:.0f}%)")

In [ ]:
#@title 🎧 Code Walkthrough: Forward Pass Tp
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_forward_pass_tp.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Forward pass: simulate tensor parallelism

# Each GPU runs the forward pass on the same replicated input
partial_outputs = [mlp(X) for mlp in tp_mlps]

for i, out in enumerate(partial_outputs):
    print(f"GPU {i} partial output shape: {out.shape}")
    print(f"  First values: {out[0, 0, :4].tolist()}")

# Allreduce: sum partial outputs
Y_tp = simulated_allreduce(partial_outputs)

print(f"\nAfter allreduce:")
print(f"Output shape: {Y_tp.shape}")
print(f"Matches reference: {torch.allclose(Y_tp, Y_ref, atol=1e-4)}")
print(f"Max difference: {(Y_tp - Y_ref).abs().max().item():.2e}")

The tensor-parallel MLP produces exactly the same output as the standard MLP. The computation was distributed across 2 simulated GPUs with only one allreduce.

---


In [ ]:
#@title 🎧 Transition: Section4 Residual Layernorm
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_section4_residual_layernorm.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 4: Adding Residual Connections and LayerNorm

In a real transformer, the MLP output is added to the input via a residual connection:

$$\text{output} = X + \text{MLP}(\text{LayerNorm}(X))$$

Both LayerNorm and the residual add operate on **replicated** data, so they require no communication.

In [ ]:
#@title 🎧 Code Walkthrough: Tp Mlp With Residual Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_tp_mlp_with_residual_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TensorParallelMLPWithResidual(nn.Module):
    """MLP sub-block of a transformer layer with tensor parallelism.
    
    Includes LayerNorm and residual connection.
    """
    
    def __init__(self, hidden_dim: int, intermediate_dim: int,
                 world_size: int, rank: int):
        super().__init__()
        self.world_size = world_size
        self.rank = rank
        
        # LayerNorm: operates on replicated data, no TP needed
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # Tensor-parallel MLP
        self.fc1 = ColumnParallelLinear(hidden_dim, intermediate_dim, world_size, rank)
        self.fc2 = RowParallelLinear(intermediate_dim, hidden_dim, world_size, rank)
    
    def forward(self, x):
        """Forward pass with residual connection.
        
        Args:
            x: Replicated input (batch, seq, hidden_dim)
        Returns:
            Partial output -- needs allreduce, then add residual!
        """
        # LayerNorm on replicated input -- same on every GPU
        residual = x
        x = self.layer_norm(x)
        
        # Column-parallel first layer
        z = self.fc1(x)
        
        # GeLU -- local
        z = F.gelu(z)
        
        # Row-parallel second layer -- returns partial sum
        y = self.fc2(z)
        
        return y, residual  # Caller: output = allreduce(y) + residual

In [ ]:
#@title 🎧 Code Walkthrough: Demonstrate Tp Mlp With Residual
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_demonstrate_tp_mlp_with_residual.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate the full MLP sub-block with residual
T = 2
h = 64
inter = 4 * h

torch.manual_seed(42)

# Create identical LayerNorm + MLP blocks for each GPU
mlp_blocks = []
for rank in range(T):
    block = TensorParallelMLPWithResidual(h, inter, T, rank).to(device)
    mlp_blocks.append(block)

# Synchronize LayerNorm parameters (they must be identical across GPUs)
for block in mlp_blocks[1:]:
    block.layer_norm.load_state_dict(mlp_blocks[0].layer_norm.state_dict())

# Forward pass
X_res = torch.randn(2, 8, h, device=device)

partials = []
residuals = []
for block in mlp_blocks:
    partial, residual = block(X_res)
    partials.append(partial)
    residuals.append(residual)

# Allreduce partial outputs
mlp_output = simulated_allreduce(partials)

# Residual connection (local -- residual is the same on every GPU)
final_output = mlp_output + residuals[0]

print(f"Input shape:        {X_res.shape}")
print(f"MLP output shape:   {mlp_output.shape}")
print(f"Final output shape: {final_output.shape}")
print(f"\nCommunication: 1 allreduce of size {mlp_output.numel() * 2} bytes")
print(f"LayerNorm: computed identically on each GPU (no communication)")
print(f"Residual add: computed locally (no communication)")

In [ ]:
#@title 🎧 Before You Start: Todo Exercise1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo_exercise1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 1: Trace the Shapes

Fill in the expected tensor shapes at each step of the tensor-parallel MLP forward pass.

In [ ]:
# TODO Exercise 1: Predict the shapes at each step
# Given: h=512, intermediate=2048, T=4, batch=2, seq=16

h_ex = 512
inter_ex = 2048
T_ex = 4
b_ex = 2
s_ex = 16

# ---- YOUR CODE HERE ----
# Fill in the shapes as tuples:
# input_shape = (?, ?, ?)          # Replicated input
# after_layernorm = (?, ?, ?)      # After LayerNorm (still replicated)
# after_fc1_per_gpu = (?, ?, ?)    # After column-parallel fc1
# after_gelu_per_gpu = (?, ?, ?)   # After GeLU (still split)
# after_fc2_per_gpu = (?, ?, ?)    # After row-parallel fc2 (partial sum)
# after_allreduce = (?, ?, ?)      # After allreduce (replicated)
# after_residual = (?, ?, ?)       # After residual add (replicated)



# ---- END YOUR CODE ----

In [ ]:
#@title 🎧 Code Walkthrough: Todo Exercise1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_todo_exercise1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Solution for Exercise 1
print("Shape trace for h=512, inter=2048, T=4, batch=2, seq=16:")
print(f"  Input (replicated):     (2, 16, 512)")
print(f"  After LayerNorm:        (2, 16, 512)  -- still replicated")
print(f"  After fc1 (per GPU):    (2, 16, 512)  -- 2048/4 = 512 per GPU")
print(f"  After GeLU (per GPU):   (2, 16, 512)  -- element-wise, same shape")
print(f"  After fc2 (per GPU):    (2, 16, 512)  -- partial sum, full output width")
print(f"  After AllReduce:        (2, 16, 512)  -- replicated, complete result")
print(f"  After residual add:     (2, 16, 512)  -- replicated")

# Verify with actual tensors
torch.manual_seed(42)
block_verify = TensorParallelMLPWithResidual(h_ex, inter_ex, T_ex, 0).to(device)
X_verify = torch.randn(b_ex, s_ex, h_ex, device=device)

with torch.no_grad():
    normed = block_verify.layer_norm(X_verify)
    after_fc1 = block_verify.fc1(normed)
    after_gelu = F.gelu(after_fc1)
    after_fc2 = block_verify.fc2(after_gelu)

print(f"\nActual shapes on GPU 0:")
print(f"  After LayerNorm: {normed.shape}")
print(f"  After fc1:       {after_fc1.shape}")
print(f"  After GeLU:      {after_gelu.shape}")
print(f"  After fc2:       {after_fc2.shape}")

---


In [ ]:
#@title 🎧 Transition: Section5 Scaling Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_section5_scaling_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 5: Scaling Test -- Different Parallelism Degrees

Let us verify correctness with T=1, 2, 4, and 8.

In [ ]:
#@title 🎧 Code Walkthrough: Test Tp Mlp
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_test_tp_mlp.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def test_tensor_parallel_mlp(hidden_dim, batch, seq_len, tp_degree):
    """Test tensor-parallel MLP correctness for a given TP degree."""
    intermediate = 4 * hidden_dim
    
    torch.manual_seed(42)
    
    # Reference: standard MLP
    mlp_std = StandardMLP(hidden_dim, intermediate).to(device)
    X_test = torch.randn(batch, seq_len, hidden_dim, device=device)
    Y_std = mlp_std(X_test)
    
    # Tensor-parallel MLP
    tp_shards = [
        TensorParallelMLP.from_standard_mlp(mlp_std, tp_degree, rank).to(device)
        for rank in range(tp_degree)
    ]
    
    # Forward pass on all shards
    partials = [shard(X_test) for shard in tp_shards]
    Y_tp = simulated_allreduce(partials)
    
    matches = torch.allclose(Y_tp, Y_std, atol=1e-4)
    max_diff = (Y_tp - Y_std).abs().max().item()
    
    params_std = sum(p.numel() for p in mlp_std.parameters())
    params_per_gpu = sum(p.numel() for p in tp_shards[0].parameters())
    
    return matches, max_diff, params_std, params_per_gpu


print(f"{'T':>4} {'Match':>8} {'Max Diff':>12} {'Full Params':>15} {'Per GPU':>12} {'Savings':>10}")
print("-" * 65)

for T in [1, 2, 4, 8]:
    match, diff, total, per_gpu = test_tensor_parallel_mlp(
        hidden_dim=128, batch=2, seq_len=16, tp_degree=T
    )
    savings = (1 - per_gpu / total) * 100
    print(f"{T:>4} {str(match):>8} {diff:>12.2e} {total:>15,} {per_gpu:>12,} {savings:>9.1f}%")

The results match for all parallelism degrees, and memory per GPU scales as expected -- each GPU stores only $1/T$ of the parameters.

---


In [ ]:
#@title 🎧 Transition: Section6 Gelu Locally
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_section6_gelu_locally.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 6: Why GeLU Can Be Applied Locally

This is a subtle but important point. GeLU is an element-wise function, so applying it to split data is the same as applying it to the full data and then splitting. Let us verify this.

In [ ]:
#@title 🎧 Code Walkthrough: Gelu Local Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_gelu_local_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Why GeLU works locally on split data
h_gelu = 16
inter_gelu = 4 * h_gelu
T_gelu = 4
batch_gelu = 2

torch.manual_seed(42)
Z_full = torch.randn(batch_gelu, inter_gelu, device=device)

# Method 1: Apply GeLU to full tensor, then split
gelu_full = F.gelu(Z_full)
gelu_then_split = torch.chunk(gelu_full, T_gelu, dim=-1)

# Method 2: Split first, then apply GeLU to each piece
Z_parts = torch.chunk(Z_full, T_gelu, dim=-1)
split_then_gelu = [F.gelu(z) for z in Z_parts]

# Compare
print("GeLU(full_tensor) then split  ==  split then GeLU(each)?")
for i in range(T_gelu):
    match = torch.allclose(gelu_then_split[i], split_then_gelu[i])
    print(f"  GPU {i}: {match}")

print(f"\nThis works because GeLU is element-wise: f(x_i) depends only on x_i.")
print(f"Contrast with softmax, which depends on ALL elements -- that cannot be split!")

# Show that softmax CANNOT be split
sm_full = F.softmax(Z_full, dim=-1)
Z_parts_sm = torch.chunk(Z_full, T_gelu, dim=-1)
sm_parts = [F.softmax(z, dim=-1) for z in Z_parts_sm]
sm_combined = torch.cat(sm_parts, dim=-1)
print(f"\nSoftmax: split-then-apply matches apply-then-split? {torch.allclose(sm_full, sm_combined)}")
print(f"(This is why attention softmax must be computed on the FULL head, not split across GPUs)")

---


In [ ]:
#@title 🎧 Transition: Section7 Communication Cost
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_section7_communication_cost.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 7: Communication Cost Profiling

Let us measure the computation time at different model sizes and compare the compute time with the estimated communication time.

In [ ]:
#@title 🎧 Code Walkthrough: Profile Mlp
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_profile_mlp.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def profile_mlp(hidden_dim, batch, seq_len, tp_degree, num_iters=50):
    """Profile tensor-parallel MLP forward pass."""
    intermediate = 4 * hidden_dim
    
    torch.manual_seed(42)
    shard = TensorParallelMLP(
        hidden_dim, intermediate, tp_degree, rank=0
    ).to(device)
    X_prof = torch.randn(batch, seq_len, hidden_dim, device=device)
    
    # Warmup
    for _ in range(5):
        _ = shard(X_prof)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Timed iterations
    start = time.perf_counter()
    for _ in range(num_iters):
        _ = shard(X_prof)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / num_iters
    
    # Estimated allreduce time
    allreduce_bytes = batch * seq_len * hidden_dim * 2  # FP16
    nvlink_bw = 300e9  # 300 GB/s
    ring_factor = 2 * (tp_degree - 1) / max(tp_degree, 1)
    allreduce_time = allreduce_bytes * ring_factor / nvlink_bw
    
    return {
        'compute_ms': elapsed * 1000,
        'allreduce_ms': allreduce_time * 1000,
        'allreduce_mb': allreduce_bytes / 1e6,
        'params_per_gpu': sum(p.numel() for p in shard.parameters()),
    }

# Profile at different sizes
configs = [
    (256, 4, 64, 2),
    (512, 4, 64, 2),
    (1024, 4, 64, 4),
    (2048, 2, 128, 4),
    (4096, 1, 256, 8),
]

print(f"{'h':>6} {'b':>4} {'s':>5} {'T':>3} {'Compute':>10} {'AllReduce':>10} {'AR Size':>10} {'Params/GPU':>12}")
print("-" * 70)
for h, b, s, T in configs:
    result = profile_mlp(h, b, s, T)
    print(f"{h:>6} {b:>4} {s:>5} {T:>3} "
          f"{result['compute_ms']:>8.3f}ms "
          f"{result['allreduce_ms']:>8.4f}ms "
          f"{result['allreduce_mb']:>8.2f}MB "
          f"{result['params_per_gpu']:>12,}")

---


In [ ]:
#@title 🎧 Transition: Section8 Backward Pass
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_section8_backward_pass.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 8: Backward Pass Verification

Let us verify that the backward pass (gradient computation) also works correctly with tensor parallelism.

In [ ]:
#@title 🎧 Code Walkthrough: Backward Pass Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_backward_pass_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Backward pass verification
T = 2
h_bw = 32
inter_bw = 4 * h_bw
batch_bw, seq_bw = 2, 4

torch.manual_seed(42)

# Reference: standard MLP backward
mlp_std_bw = StandardMLP(h_bw, inter_bw).to(device)
X_bw = torch.randn(batch_bw, seq_bw, h_bw, device=device)
Y_std_bw = mlp_std_bw(X_bw)
loss_std = Y_std_bw.sum()
loss_std.backward()

# Store reference gradients
ref_grad_fc1 = mlp_std_bw.fc1.weight.grad.clone()
ref_grad_fc2 = mlp_std_bw.fc2.weight.grad.clone()

# Tensor-parallel backward
tp_shards_bw = [
    TensorParallelMLP.from_standard_mlp(mlp_std_bw, T, rank).to(device)
    for rank in range(T)
]

# Zero grads in reference before re-running
mlp_std_bw.zero_grad()

# Forward
X_bw_tp = X_bw.clone().detach()
partials_bw = [shard(X_bw_tp) for shard in tp_shards_bw]
Y_tp_bw = simulated_allreduce(partials_bw)
loss_tp = Y_tp_bw.sum()
loss_tp.backward()

# Compare gradients
print("Gradient verification (T=2):")
print(f"  Loss matches: {torch.allclose(loss_std, loss_tp, atol=1e-4)}")

# fc1 gradient: column-parallel, so gradient is also split along columns
# nn.Linear weight is (out, in), our weight is (in, out/T)
for rank in range(T):
    shard_grad = tp_shards_bw[rank].fc1.weight.grad
    print(f"  GPU {rank} fc1 grad shape: {shard_grad.shape}")

# fc2 gradient: row-parallel, so gradient is split along rows
for rank in range(T):
    shard_grad = tp_shards_bw[rank].fc2.weight.grad
    print(f"  GPU {rank} fc2 grad shape: {shard_grad.shape}")

print(f"\nEach GPU updates only its shard of the weights -- no gradient allreduce needed!")
print(f"(This is different from data parallelism, which DOES need gradient allreduce.)")

In [ ]:
#@title 🎧 Before You Start: Todo Exercise2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_todo_exercise2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Complete MLP with Residual and Backward

Build a complete tensor-parallel MLP sub-block with LayerNorm and residual connection, run forward and backward, and verify the loss gradient is correct.

In [ ]:
# TODO Exercise 2: Full MLP sub-block with backward

T_ex2 = 2
h_ex2 = 32
inter_ex2 = 4 * h_ex2

torch.manual_seed(42)
X_ex2 = torch.randn(2, 4, h_ex2, device=device, requires_grad=True)

# ---- YOUR CODE HERE ----
# 1. Create a standard MLP and compute reference output with residual:
#    Y_ref = X + MLP(LayerNorm(X))
# 2. Compute reference loss = Y_ref.sum() and backward
# 3. Create T=2 TensorParallelMLPWithResidual blocks
#    (remember to sync LayerNorm params!)
# 4. Forward each shard, allreduce, add residual
# 5. Compute loss and backward
# 6. Verify losses match



# ---- END YOUR CODE ----

In [ ]:
#@title 🎧 Code Walkthrough: Todo Exercise2 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_todo_exercise2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Solution for Exercise 2

# Reference
torch.manual_seed(42)
X_ex2 = torch.randn(2, 4, h_ex2, device=device)

ln_ref = nn.LayerNorm(h_ex2).to(device)
mlp_ref_ex2 = StandardMLP(h_ex2, inter_ex2).to(device)

Y_ref_ex2 = X_ex2 + mlp_ref_ex2(ln_ref(X_ex2))
loss_ref_ex2 = Y_ref_ex2.sum()
loss_ref_ex2.backward()
print(f"Reference loss: {loss_ref_ex2.item():.6f}")

# Tensor-parallel version
tp_blocks_ex2 = []
for rank in range(T_ex2):
    block = TensorParallelMLPWithResidual(h_ex2, inter_ex2, T_ex2, rank).to(device)
    # Copy LayerNorm params from reference
    block.layer_norm.load_state_dict(ln_ref.state_dict())
    # Copy weight shards from reference MLP
    block.fc1 = ColumnParallelLinear.from_linear(mlp_ref_ex2.fc1, T_ex2, rank)
    block.fc2 = RowParallelLinear.from_linear(mlp_ref_ex2.fc2, T_ex2, rank)
    tp_blocks_ex2.append(block)

# Forward
partials_ex2 = []
residuals_ex2 = []
for block in tp_blocks_ex2:
    p, r = block(X_ex2.detach())
    partials_ex2.append(p)
    residuals_ex2.append(r)

Y_tp_ex2 = simulated_allreduce(partials_ex2) + residuals_ex2[0]
loss_tp_ex2 = Y_tp_ex2.sum()

print(f"TP loss:        {loss_tp_ex2.item():.6f}")
print(f"Losses match:   {torch.allclose(loss_ref_ex2, loss_tp_ex2, atol=1e-4)}")
print(f"Outputs match:  {torch.allclose(Y_ref_ex2, Y_tp_ex2, atol=1e-4)}")

---


In [ ]:
#@title 🎧 Transition: Section9 Memory Analysis
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_section9_memory_analysis.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 9: Memory Savings Analysis for Real Models

Let us calculate memory savings for realistic transformer configurations.

In [ ]:
#@title 🎧 Code Walkthrough: Analyze Memory
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_27_analyze_memory.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analyze_memory(model_name, hidden_dim, num_layers, num_heads, tp_degree):
    """Analyze memory savings from tensor parallelism for an MLP block."""
    inter = 4 * hidden_dim
    
    # Weight parameters per layer (MLP only)
    mlp_params = hidden_dim * inter + inter * hidden_dim  # W1 + W2
    
    # Attention parameters per layer
    attn_params = 4 * hidden_dim * hidden_dim  # Q, K, V, O
    
    total_params_per_layer = mlp_params + attn_params
    total_params = total_params_per_layer * num_layers
    
    # Memory per GPU
    params_per_gpu = total_params // tp_degree
    
    # In FP16: 2 bytes per parameter
    # With Adam optimizer: weights + gradients + 2 optimizer states = 16 bytes/param (mixed precision)
    weight_mem_gb = total_params * 2 / 1e9
    weight_per_gpu_gb = params_per_gpu * 2 / 1e9
    training_mem_gb = total_params * 16 / 1e9
    training_per_gpu_gb = params_per_gpu * 16 / 1e9
    
    print(f"\n{'='*50}")
    print(f"{model_name} (h={hidden_dim}, L={num_layers}, T={tp_degree})")
    print(f"{'='*50}")
    print(f"Total parameters:     {total_params/1e9:.1f}B")
    print(f"Weight memory (FP16): {weight_mem_gb:.1f} GB total, {weight_per_gpu_gb:.1f} GB/GPU")
    print(f"Training memory:      {training_mem_gb:.1f} GB total, {training_per_gpu_gb:.1f} GB/GPU")
    print(f"Memory reduction:     {tp_degree}x")
    
    # Check if it fits on an 80GB A100
    a100_mem = 80
    fits = training_per_gpu_gb < a100_mem * 0.7  # Leave 30% for activations
    print(f"Fits on A100 80GB:    {'Yes' if fits else 'No'} (need ~{training_per_gpu_gb:.1f}GB + activations)")

# Analyze different models
models = [
    ('LLaMA 7B',   4096, 32, 32, 1),
    ('LLaMA 7B',   4096, 32, 32, 2),
    ('LLaMA 13B',  5120, 40, 40, 2),
    ('LLaMA 13B',  5120, 40, 40, 4),
    ('LLaMA 70B',  8192, 80, 64, 4),
    ('LLaMA 70B',  8192, 80, 64, 8),
    ('GPT-3 175B', 12288, 96, 96, 8),
]

for name, h, L, heads, T in models:
    analyze_memory(name, h, L, heads, T)

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_28_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


---
## Summary

In this notebook, you built a complete Megatron-LM style tensor-parallel MLP block:

1. **Column-parallel first layer** splits the output dimension, producing split intermediate activations
2. **GeLU activation** runs locally because it is element-wise
3. **Row-parallel second layer** consumes the split activations and produces partial sums
4. **One allreduce** combines the partial sums to get the full output
5. **Residual add and LayerNorm** operate on replicated data -- no communication

The total communication is **one allreduce per MLP block**. This is the minimal possible communication for splitting a two-layer network.

In the next notebook, we will apply the same strategy to **multi-head self-attention**, where the parallelism is even more natural because attention heads are already independent.